# PIE: Architecture and Internal Representations

**Notebook 01 — Architecture Inspection**

## Objective

This notebook examines the architecture and internal computation
of the pretrained PIE (Perturbation Is Everything) model.

The primary objective is to identify suitable activation extraction
points for subsequent layer-wise Sparse Autoencoder (SAE) analysis.

## Research Questions

1. How does PIE process biological information?
2. How are the encoder, processor, and decoder organized?
3. What types of internal representations are produced?
4. What are the dimensions and meanings of these representations?
5. Which internal activations are suitable for SAE training?

## Methodology

We examine the official PIE implementation and pretrained model.

The analysis includes:
- Model configuration and architecture inspection.
- Examination of the encoder–processor–decoder computation flow.
- Identification of candidate activation extraction points.
- Verification of model initialization and pretrained weights.

## References

1. Verma et al. (2026).
   *PIE: Generalizing perturbation effects across unseen
   perturbations, contexts and datasets.*

2. Kendiukhov (2026).
   *Sparse autoencoders reveal organized biological knowledge
   but minimal regulatory logic in single-cell foundation models:
   a comparative atlas of Geneformer and scGPT.*

## Scope

This notebook focuses on architecture inspection and model setup.

Large-scale activation extraction, SAE training, feature
statistics, and cross-layer analysis are addressed in
subsequent notebooks.

In [1]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: not available")

Python: 3.12.13
Interpreter: d:\bio_paper\pie-sae\.venv\Scripts\python.exe
PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA GeForce GTX 1650


## 1. Project Environment and Model Configuration

The analysis uses an isolated Python environment and the
official PIE and bio-sae repositories.

The external implementations are included as Git submodules,
ensuring that the project does not depend on other local
research directories.

### Pretrained Model

We begin with the official PIE checkpoint corresponding to
the K562 held-out context fold of the Replogle–Nadig dataset.

This checkpoint is used to establish and validate the
architecture inspection and activation extraction pipeline.

The model configuration is loaded from `config.yaml`.

### Architecture Parameters

The principal configuration parameters are:

- `d_model`: hidden representation dimensionality.
- `n_latents`: number of learned latent slots.
- `n_encoder_layers`: number of encoder blocks.
- `n_processor_layers`: number of processor blocks.
- `n_decoder_layers`: number of decoder blocks.
- `num_heads`: number of attention heads.

These parameters determine the structure of the pretrained
model and the expected dimensions of internal activations.

In [2]:

from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    
import os

os.environ["PIE_DATA_ROOT"] = str(PROJECT_ROOT / "data")
os.environ["PIE_CACHE_DIR"] = str(PROJECT_ROOT / "cache")
os.environ["PIE_RUNS_ROOT"] = str(PROJECT_ROOT / "runs")

PIE_ROOT = PROJECT_ROOT / "external" / "pie"
CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "models"
    / "replogle_wdataset"
    / "k562"
)

assert PIE_ROOT.is_dir(), f"PIE source not found: {PIE_ROOT}"
assert (CHECKPOINT_DIR / "best_auprc.ckpt").is_file()

with open(CHECKPOINT_DIR / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

architecture = config["model"]

print("PIE source:", PIE_ROOT)
print("Checkpoint:", CHECKPOINT_DIR)
print("Fold:", config["vars"]["fold"])
print()

for key in [
    "d_model",
    "n_latents",
    "n_encoder_layers",
    "n_processor_layers",
    "n_decoder_layers",
    "num_heads",
]:
    print(f"{key}: {architecture[key]}")


PIE source: D:\bio_paper\pie-sae\external\pie
Checkpoint: D:\bio_paper\pie-sae\models\replogle_wdataset\k562
Fold: k562

d_model: 768
n_latents: 512
n_encoder_layers: 4
n_processor_layers: 6
n_decoder_layers: 4
num_heads: 8


## 2. PIE Architecture

PIE is based on the Perceiver IO architecture.

Its computation is organized into three principal stages:
encoder, processor, and decoder.

### 2.1 Encoder

The encoder integrates biological knowledge sources into
a set of learned latent representations.

It uses cross-attention between latent queries and
the input source representations.

For the selected checkpoint:

- Number of encoder blocks: 4.
- Number of latent slots: 512.
- Hidden dimension: 768.

### 2.2 Processor

The processor transforms the latent representations
using self-attention.

Each latent slot can exchange information with other
latent slots through the attention mechanism.

The selected checkpoint contains 6 processor blocks.

### 2.3 Decoder

The decoder uses gene-specific queries to retrieve information
from the processed latent memory.

Gene queries are constructed using gene representations
and available experimental evidence.

The selected checkpoint contains 4 decoder blocks.

### 2.4 Prediction Heads

PIE produces three types of predictions:

1. Differential expression (DE).
2. Log fold change (LFC).
3. Changes in pseudobulk gene expression.

### 2.5 Representation Types

An important architectural distinction is that encoder and
processor activations correspond to latent slots, whereas
decoder activations correspond to output genes.

These representations have different semantic roles,
even when their hidden dimensionalities are identical.

Therefore, their SAE features should initially be analyzed
within their respective representation types.

### SAE Analysis

We identify the outputs of all 14 Transformer blocks as
candidate extraction points for layer-wise SAE training.

The final selection will be verified using actual model
forward passes and activation tensor shapes.

In [3]:

import sys
import inspect

PIE_SRC = PIE_ROOT / "src"

if str(PIE_SRC) not in sys.path:
    sys.path.insert(0, str(PIE_SRC))

from pie.model.pie import PieModel

print("PIE model class:", PieModel)
print("Source file:", inspect.getfile(PieModel))

print("\nConstructor:")
print(inspect.signature(PieModel.__init__))

print("\nMain methods:")
for name in ["forward", "_encode", "_build_kv", "_decode_group", "_decode_chunk"]:
    method = getattr(PieModel, name)
    print(f"{name}{inspect.signature(method)}")


PIE model class: <class 'pie.model.pie.PieModel'>
Source file: D:\bio_paper\pie-sae\external\pie\src\pie\model\pie.py

Constructor:
(self, cfg: 'ModelConfig', stats: 'DataStats', source_dims: 'dict[str, int]', gene_query_text: 'Tensor') -> 'None'

Main methods:
forward(self, batch: 'Batch') -> 'list[GroupOutput]'
_encode(self, batch: 'Batch') -> 'Tensor'
_build_kv(self, source_tokens: 'Mapping[str, Tensor]', source_masks: 'Mapping[str, Tensor]', b: 'int') -> 'tuple[Tensor, Tensor]'
_decode_group(self, memory: 'Tensor', group: 'GeneGroup') -> 'GroupOutput'
_decode_chunk(self, memory: 'Tensor', gene_ids: 'Tensor', ctrl_means: 'Tensor', evidence: 'Mapping[str, Tensor]') -> 'tuple[Tensor, Tensor]'


d:\bio_paper\pie-sae\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Loading and Inspecting the Pretrained Checkpoint

We load the official pretrained PIE checkpoint using
the original `load_checkpoint` function.

At this stage, we inspect the checkpoint contents
without performing model inference.

### Verification

The inspection includes:

1. Successful checkpoint loading.
2. Restored architecture configuration.
3. Number of parameter tensors.
4. Total number of stored parameters.
5. Names and dimensions of selected parameter tensors.

### Parameters vs. Activations

A model parameter is a value learned during training.

An activation is an intermediate representation produced
when the model processes a particular input.

Sparse Autoencoders will be trained on internal activations,
not directly on the model's parameter matrices.

### Next Step

After inspecting the checkpoint, we will reconstruct
the pretrained model and verify its forward computation.

We will then identify and validate the activation
extraction points for all selected Transformer blocks.

In [4]:
import os

os.environ["PIE_RUNS_ROOT"] = str(PROJECT_ROOT / "runs")
from pie.predict import load_checkpoint

checkpoint_path = CHECKPOINT_DIR / "best_auprc.ckpt"

loaded = load_checkpoint(checkpoint_path)

print("Checkpoint loaded successfully!")
print("Checkpoint:", checkpoint_path.name)

print("\nModel configuration:")
print("Model width:", loaded.config.model.d_model)
print("Latent slots:", loaded.config.model.n_latents)
print("Encoder layers:", loaded.config.model.n_encoder_layers)
print("Processor layers:", loaded.config.model.n_processor_layers)
print("Decoder layers:", loaded.config.model.n_decoder_layers)

print("\nState dictionary:")
print("Number of tensors:", len(loaded.state_dict))

n_parameters = sum(
    tensor.numel()
    for tensor in loaded.state_dict.values()
)

print(f"Stored parameters: {n_parameters:,}")

print("\nFirst 10 parameter tensors:")

for name, tensor in list(loaded.state_dict.items())[:10]:
    print(f"{name}: {tuple(tensor.shape)}")

Checkpoint loaded successfully!
Checkpoint: best_auprc.ckpt

Model configuration:
Model width: 768
Latent slots: 512
Encoder layers: 4
Processor layers: 6
Decoder layers: 4

State dictionary:
Number of tensors: 249
Stored parameters: 79,106,918

First 10 parameter tensors:
latents: (512, 768)
source_projections.esm2.0.weight: (1280,)
source_projections.esm2.1.weight: (768, 1280)
source_projections.ncbi_text.0.weight: (2048,)
source_projections.ncbi_text.1.weight: (768, 2048)
source_projections.string_space.0.weight: (512,)
source_projections.string_space.1.weight: (768, 512)
source_projections.depmap_gene_effect.0.weight: (1186,)
source_projections.depmap_gene_effect.1.weight: (768, 1186)
source_projections.context_text.0.weight: (3072,)


In [5]:
import os
import shutil

# Local project directories
directories = {
    "PIE_DATA_ROOT": PROJECT_ROOT / "data",
    "PIE_CACHE_DIR": PROJECT_ROOT / "cache",
    "PIE_RUNS_ROOT": PROJECT_ROOT / "runs",
}

for variable, path in directories.items():
    path.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(path.resolve())

    print(f"{variable}: {path.resolve()}")

# Check available disk space
total, used, free = shutil.disk_usage(PROJECT_ROOT)

print(f"\nAvailable disk space: {free / 1024**3:.2f} GiB")

# Inspect required data sources
data_cfg = loaded.config.data

print("\nPreprocessed datasets:")
print(data_cfg.preprocessed_dirs)

print("\nKnowledge sources:")
for name, path in data_cfg.source_dirs.items():
    print(f"{name}: {path}")

print("\nGene text embeddings:")
print(data_cfg.gene_text_dir)

PIE_DATA_ROOT: D:\bio_paper\pie-sae\data
PIE_CACHE_DIR: D:\bio_paper\pie-sae\cache
PIE_RUNS_ROOT: D:\bio_paper\pie-sae\runs

Available disk space: 186.85 GiB

Preprocessed datasets:
['hf://datasets/arcinstitute/pie_replogle_nadig_essential@20c9faef76fc96fdc809871340bd93499413dfe7/preprocessed']

Knowledge sources:
esm2: hf://datasets/arcinstitute/pie_sources@cb1aaa4e7655605bdc70a9bd77bbd62016b8c7d7/esm2
ncbi_text: hf://datasets/arcinstitute/pie_sources@cb1aaa4e7655605bdc70a9bd77bbd62016b8c7d7/ncbi_text
string_space: hf://datasets/arcinstitute/pie_sources@cb1aaa4e7655605bdc70a9bd77bbd62016b8c7d7/string_space
depmap_gene_effect: hf://datasets/arcinstitute/pie_sources@cb1aaa4e7655605bdc70a9bd77bbd62016b8c7d7/depmap_gene_effect
context_text: hf://datasets/arcinstitute/pie_sources@cb1aaa4e7655605bdc70a9bd77bbd62016b8c7d7/context_text
perturbation_text: hf://datasets/arcinstitute/pie_sources@cb1aaa4e7655605bdc70a9bd77bbd62016b8c7d7/perturbation_text

Gene text embeddings:
hf://datasets/arcin

## 4. Initializing the PIE Data Module

We initialize the official PIE data module using the
configuration and dataset statistics restored from the
pretrained checkpoint.

The data module resolves the biological knowledge sources
and prepares the information required for model inference.

The original sources are retrieved from version-pinned
Hugging Face repositories.

Large embedding arrays are accessed using memory mapping.

All project-specific data and caches are stored within
the independent `pie-sae` project.

In [6]:
from pie.data.datamodule import PieDataModule

train_data = loaded.config.data

dm = PieDataModule(
    cfg=train_data,
    run_dir=None,
    stats=loaded.stats,
    seed=loaded.config.seed,
    evidence_source=train_data,
)

print("Data module initialized.")
print("Data configuration:", type(dm.cfg).__name__)
print("Dataset statistics:", type(dm.stats).__name__)

Data module initialized.
Data configuration: DataConfig
Dataset statistics: DataStats


## 5. Preparing Model Inputs

The official PIE data module has been initialized successfully.

We now prepare the dataset-dependent information required
to reconstruct the pretrained model:

- Dimensions of the biological knowledge sources.
- Gene-query text embeddings.
- Dataset statistics associated with the checkpoint.

These components must be compatible with the pretrained
model parameters.

The original PIE implementation is used to ensure
consistent model initialization.

In [7]:
# Prepare dataset statistics and source metadata
dm.setup_stats(rank=0)

# Retrieve source dimensions and gene embeddings
source_dims = dict(dm.source_dims)
gene_query_text = dm.gene_query_text()

print("Data module setup completed.")

print("\nKnowledge source dimensions:")
for name, dim in source_dims.items():
    print(f"{name}: {dim}")

print("\nGene query embeddings:")
print("Shape:", tuple(gene_query_text.shape))
print("Dtype:", gene_query_text.dtype)

Data module setup completed.

Knowledge source dimensions:
esm2: 1280
ncbi_text: 2048
string_space: 512
depmap_gene_effect: 1186
context_text: 3072
perturbation_text: 3072

Gene query embeddings:
Shape: (6642, 3072)
Dtype: torch.float32


## 6. Reconstructing the Pretrained PIE Model

We reconstruct the pretrained PIE model using the official
architecture implementation and the previously loaded checkpoint.

The model requires four components:

1. Model configuration.
2. Dataset statistics.
3. Knowledge-source dimensions.
4. Gene-query text embeddings.

### Model Initialization

The `PieModel` constructor creates the complete neural network
architecture, including the encoder, processor, decoder,
and prediction heads.

### Pretrained Weight Restoration

The original pretrained parameters are restored using
`load_state_dict(..., strict=True)`.

Strict loading ensures that the checkpoint parameters match
the reconstructed model architecture.

### Verification

We verify:

- Successful restoration of pretrained parameters.
- Absence of missing or unexpected parameter keys.
- Correct number of Transformer blocks.
- Total number of model parameters.
- Evaluation mode and device placement.

The model is initially loaded on the CPU.

In [8]:
# Reconstruct the pretrained PIE model

model = PieModel(
    cfg=loaded.config.model,
    stats=loaded.stats,
    source_dims=source_dims,
    gene_query_text=gene_query_text,
)

# Restore pretrained parameters
load_result = model.load_state_dict(
    loaded.state_dict,
    strict=True,
)

# Freeze the pretrained model and enable evaluation mode
model.requires_grad_(False)
model.eval()

print("Pretrained PIE model reconstructed successfully!")

# Verify checkpoint compatibility
print("\nCheckpoint verification:")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

# Inspect architecture
print("\nModel architecture:")
print("Encoder blocks:", len(model.encoder))
print("Processor blocks:", len(model.processor))
print("Decoder blocks:", len(model.decoder))

# Count model parameters
total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nModel parameters:")
print(f"Total parameters: {total_parameters:,}")
print(f"Parameters with requires_grad=True: {trainable_parameters:,}")

# Verify model state
print("\nModel state:")
print("Training mode:", model.training)
print("Device:", next(model.parameters()).device)

Pretrained PIE model reconstructed successfully!

Checkpoint verification:
Missing keys: []
Unexpected keys: []

Model architecture:
Encoder blocks: 4
Processor blocks: 6
Decoder blocks: 4

Model parameters:
Total parameters: 79,106,918
Parameters with requires_grad=True: 0

Model state:
Training mode: False
Device: cpu


## 7. Loading the Official Dataset Splits

The pretrained PIE checkpoint references version-pinned
dataset splits stored on Hugging Face.

These references use the `hf://` URI scheme and must not
be interpreted as ordinary local filesystem paths.

We use the official PIE data module to prepare the
training and validation datasets.

The held-out test data will be reserved for subsequent
evaluation of the SAE analysis.

In [9]:
# Inspect the official split reference
print("Official split reference:")
print(dm.cfg.split_dir)

# Prepare training and validation datasets
dm.setup(stage="fit")

print("\nDataset preparation completed.")

print("Training conditions:", len(dm.train_dataset))
print("Validation conditions:", len(dm.val_dataset))

Official split reference:
hf://datasets/arcinstitute/PIE_splits@396ab9563175ee887750c9eed7ccaea6f5fdbf50/replogle_wdataset/unseen_ctx/k562

Dataset preparation completed.
Training conditions: 2603
Validation conditions: 290


## 8. Inspecting the First Experimental Batch

We obtain a single experimental condition from the official
PIE training dataset.

The objective is to verify the structure of the model input
before performing inference and extracting internal activations.

A single-condition batch is used to limit memory consumption
during the initial architecture inspection.

In [10]:
# Create the official training data loader
train_loader = dm.train_dataloader()

print("Training dataset size:", len(dm.train_dataset))
print("Data loader:", type(train_loader).__name__)

# Select one experimental condition
sample = dm.train_dataset[0]

print("\nSingle dataset item:")
print("Type:", type(sample).__name__)

# Construct a batch using PIE's official collate function
batch = train_loader.collate_fn([sample])

print("\nConstructed batch:")
print("Type:", type(batch).__name__)
print("Number of gene groups:", len(batch.groups))

print("\nInput components:")
print("Source names:", list(batch.source_tokens.keys()))

for name, tensor in batch.source_tokens.items():
    print(f"{name}: {tuple(tensor.shape)}")

print("\nGene groups:")
for index, group in enumerate(batch.groups):
    print(
        f"Group {index}: "
        f"gene_ids shape = {tuple(group.gene_ids.shape)}"
    )

Training dataset size: 2603
Data loader: DataLoader

Single dataset item:
Type: dict

Constructed batch:
Type: Batch
Number of gene groups: 1

Input components:
Source names: ['context_text', 'depmap_gene_effect', 'esm2', 'ncbi_text', 'perturbation_text', 'string_space']
context_text: (1, 1, 3072)
depmap_gene_effect: (1, 1, 1186)
esm2: (1, 1, 1280)
ncbi_text: (1, 475, 2048)
perturbation_text: (1, 1, 3072)
string_space: (1, 1, 512)

Gene groups:
Group 0: gene_ids shape = (6642,)


## 9. First Forward Pass

We perform the first inference pass using the pretrained PIE
model and a single experimental condition from the official
training dataset.

### Objectives

1. Verify that the reconstructed model accepts the official Batch.
2. Confirm that all prediction heads produce valid outputs.
3. Inspect the dimensions of DE, LFC, and delta-p predictions.
4. Measure inference time.

The original PIE parameters remain frozen.

We use `torch.inference_mode()` to disable gradient tracking
during inference.

This experiment validates the computational pipeline before
registering activation extraction hooks.

In [11]:
import time
import torch

from pie.predict import _batch_to

# Use CPU and float32 for the initial verification
device = torch.device("cpu")

model = model.to(device)
model.eval()

# Convert the batch using the official PIE utility
batch_ready = _batch_to(
    batch,
    device,
    model.latents.dtype,
)

print("Model device:", model.latents.device)
print("Model dtype:", model.latents.dtype)

print("\nInput dtypes:")
for name, tensor in batch_ready.source_tokens.items():
    print(f"{name}: {tensor.dtype}")

# Run inference
start_time = time.perf_counter()

with torch.inference_mode():
    outputs = model(batch_ready)

elapsed = time.perf_counter() - start_time

print("\nForward pass completed successfully!")
print(f"Inference time: {elapsed:.2f} seconds")

print("\nNumber of output groups:", len(outputs))

# Inspect the first output group
out = outputs[0]

print("\nPrediction shapes:")
print("DE logits:", tuple(out.de_logits.shape))
print("LFC:", tuple(out.lfc.shape))
print("Delta-p logits:", tuple(out.dp_logits.shape))

print("\nNumerical validity:")
print("DE finite:", torch.isfinite(out.de_logits).all().item())
print("LFC finite:", torch.isfinite(out.lfc).all().item())
print("Delta-p finite:", torch.isfinite(out.dp_logits).all().item())

Model device: cpu
Model dtype: torch.float32

Input dtypes:
context_text: torch.float32
depmap_gene_effect: torch.float32
esm2: torch.float32
ncbi_text: torch.float32
perturbation_text: torch.float32
string_space: torch.float32

Forward pass completed successfully!
Inference time: 1.98 seconds

Number of output groups: 1

Prediction shapes:
DE logits: (1, 6642, 2)
LFC: (1, 6642)
Delta-p logits: (1, 6642, 17)

Numerical validity:
DE finite: True
LFC finite: True
Delta-p finite: True


## 10. Inspecting Internal Activations with Forward Hooks

### Objective

We inspect the outputs of all encoder, processor, and decoder
Transformer blocks in the pretrained PIE model.

### Method

PyTorch forward hooks are registered on the outputs of:

- 4 encoder blocks.
- 6 processor blocks.
- 4 decoder blocks.

The hooks record activation shapes, data types, and small
numerical samples without modifying the model computations.

### Verification

We verify that:

1. All 14 Transformer blocks produce activations.
2. Encoder and processor outputs correspond to latent slots.
3. Decoder outputs correspond to gene-specific representations.
4. Activation dimensions match the model configuration.
5. Forward hooks do not change the model predictions.

The complete activation extraction pipeline will be
implemented in the next notebook.

In [12]:
import torch

# Store activation metadata
activation_records = {}

# Keep track of registered hooks
handles = []


def make_hook(layer_name):
    """Record output metadata without modifying activations."""

    def hook(module, inputs, output):

        if not isinstance(output, torch.Tensor):
            raise TypeError(
                f"Expected Tensor at {layer_name}, "
                f"got {type(output)}"
            )

        record = {
            "shape": tuple(output.shape),
            "dtype": str(output.dtype),
            "sample": (
                output[0, 0, :5]
                .detach()
                .cpu()
                .tolist()
            ),
        }

        activation_records.setdefault(
            layer_name, []
        ).append(record)

    return hook


# Register hooks on every Transformer block
for group_name in ["encoder", "processor", "decoder"]:

    blocks = getattr(model, group_name)

    for index, block in enumerate(blocks):

        layer_name = f"{group_name}.{index}"

        handle = block.register_forward_hook(
            make_hook(layer_name)
        )

        handles.append(handle)


# Run inference with hooks
try:
    with torch.inference_mode():
        hooked_outputs = model(batch_ready)

finally:
    # Always remove hooks after the experiment
    for handle in handles:
        handle.remove()


# Display activation information
print("ACTIVATION EXTRACTION RESULTS")
print("=" * 60)

for layer_name, records in activation_records.items():

    print(f"\nLayer: {layer_name}")
    print(f"Number of calls: {len(records)}")

    for i, record in enumerate(records):
        print(
            f"  Call {i + 1}: "
            f"shape={record['shape']}, "
            f"dtype={record['dtype']}"
        )

    print("  First five values:", records[0]["sample"])


# Verify model predictions remain unchanged
print("\n" + "=" * 60)
print("PREDICTION CONSISTENCY")

baseline = outputs[0]
hooked = hooked_outputs[0]

for name in ["de_logits", "lfc", "dp_logits"]:

    original = getattr(baseline, name)
    modified = getattr(hooked, name)

    max_difference = (
        (original - modified)
        .abs()
        .max()
        .item()
    )

    print(f"{name}: max absolute difference = {max_difference:.8f}")


print("\nTotal observed blocks:", len(activation_records))

ACTIVATION EXTRACTION RESULTS

Layer: encoder.0
Number of calls: 1
  Call 1: shape=(1, 512, 768), dtype=torch.float32
  First five values: [-0.03048884868621826, -0.8191884160041809, 1.0315455198287964, -0.18688255548477173, 0.9189960956573486]

Layer: encoder.1
Number of calls: 1
  Call 1: shape=(1, 512, 768), dtype=torch.float32
  First five values: [1.000281572341919, -0.6991339325904846, 0.5986883640289307, -0.5299181342124939, 1.9731236696243286]

Layer: encoder.2
Number of calls: 1
  Call 1: shape=(1, 512, 768), dtype=torch.float32
  First five values: [0.8156793713569641, -0.1828669160604477, -0.029549114406108856, -1.0455390214920044, 1.880998969078064]

Layer: encoder.3
Number of calls: 1
  Call 1: shape=(1, 512, 768), dtype=torch.float32
  First five values: [-0.43080416321754456, -1.7539104223251343, -1.1172996759414673, -0.607998788356781, 0.7585821151733398]

Layer: processor.0
Number of calls: 1
  Call 1: shape=(1, 512, 768), dtype=torch.float32
  First five values: [-0.3

## 11. Activation Extraction Verification

We summarize the activation metadata collected from all
14 Transformer blocks.

The verification checks:

1. The expected number of encoder, processor, and decoder blocks.
2. The number of forward calls for each block.
3. The dimensions of the extracted representations.
4. Consistency of model predictions before and after hook registration.

This experiment validates the selected extraction points
before implementing the full layer-wise SAE pipeline.

In [13]:
import torch

print("LAYER-WISE ACTIVATION SUMMARY")
print("=" * 75)

for group_name in ["encoder", "processor", "decoder"]:
    blocks = getattr(model, group_name)

    print(f"\n{group_name.upper()}")

    for index in range(len(blocks)):
        name = f"{group_name}.{index}"
        records = activation_records[name]

        shapes = [record["shape"] for record in records]

        print(
            f"{name:<15} "
            f"calls={len(records):<2} "
            f"shapes={shapes}"
        )

        assert all(shape[-1] == 768 for shape in shapes)

        if group_name in ["encoder", "processor"]:
            assert len(records) == 1
            assert shapes[0] == (1, 512, 768)

        if group_name == "decoder":
            assert len(records) == 4
            assert shapes == [
                (1, 2048, 768),
                (1, 2048, 768),
                (1, 2048, 768),
                (1, 498, 768),
            ]

assert len(activation_records) == 14

print("\nPREDICTION CONSISTENCY")
print("=" * 75)

for name in ["de_logits", "lfc", "dp_logits"]:
    baseline_tensor = getattr(outputs[0], name)
    hooked_tensor = getattr(hooked_outputs[0], name)

    max_diff = (
        (baseline_tensor - hooked_tensor)
        .abs()
        .max()
        .item()
    )

    print(f"{name}: max difference = {max_diff:.10f}")

    assert torch.equal(
        baseline_tensor,
        hooked_tensor,
    )

print("\nAll activation and prediction checks passed!")

LAYER-WISE ACTIVATION SUMMARY

ENCODER
encoder.0       calls=1  shapes=[(1, 512, 768)]
encoder.1       calls=1  shapes=[(1, 512, 768)]
encoder.2       calls=1  shapes=[(1, 512, 768)]
encoder.3       calls=1  shapes=[(1, 512, 768)]

PROCESSOR
processor.0     calls=1  shapes=[(1, 512, 768)]
processor.1     calls=1  shapes=[(1, 512, 768)]
processor.2     calls=1  shapes=[(1, 512, 768)]
processor.3     calls=1  shapes=[(1, 512, 768)]
processor.4     calls=1  shapes=[(1, 512, 768)]
processor.5     calls=1  shapes=[(1, 512, 768)]

DECODER
decoder.0       calls=4  shapes=[(1, 2048, 768), (1, 2048, 768), (1, 2048, 768), (1, 498, 768)]
decoder.1       calls=4  shapes=[(1, 2048, 768), (1, 2048, 768), (1, 2048, 768), (1, 498, 768)]
decoder.2       calls=4  shapes=[(1, 2048, 768), (1, 2048, 768), (1, 2048, 768), (1, 498, 768)]
decoder.3       calls=4  shapes=[(1, 2048, 768), (1, 2048, 768), (1, 2048, 768), (1, 498, 768)]

PREDICTION CONSISTENCY
de_logits: max difference = 0.0000000000
lfc: max dif

## 12. Conclusions

The pretrained PIE model was successfully reconstructed and validated using the official implementation and checkpoint.

### Main Results

- The model contains 79,106,918 parameters.
- All pretrained parameters were restored successfully with strict state-dictionary loading.
- The architecture consists of 4 encoder, 6 processor, and 4 decoder Transformer blocks.
- The hidden representation dimension is 768.
- Encoder and processor blocks produce representations of shape `(1, 512, 768)` for a single experimental condition.
- Decoder blocks produce gene-specific representations in four chunks containing 2,048, 2,048, 2,048, and 498 genes.
- Forward hooks successfully captured activation metadata from all 14 Transformer blocks.
- Model predictions remained exactly unchanged after registering the hooks.

### Interpretation

The results confirm that the selected extraction points provide access to the internal representations of all principal Transformer blocks without modifying the pretrained model's predictions.

An important architectural distinction was identified: encoder and processor representations correspond to latent slots, whereas decoder representations correspond to output genes.

This distinction must be preserved during subsequent SAE training and cross-layer analysis.

### Limitations

The current verification was performed using a single experimental condition.

The experiment validates activation extraction but does not establish the biological interpretability or statistical properties of the extracted representations.

### Next Steps

The next notebook will implement a scalable activation extraction pipeline for the training and validation datasets.

The resulting activation datasets will be used to train separate sparse autoencoders for each selected PIE layer.